In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import requests
import timm
import torch
import torch.nn.functional as F
from torchvision import models, datasets, tv_tensors
from torchvision import transforms as T
from torchvision.utils import make_grid
import types
import albumentations as A
import seaborn as sns

from PIL import Image
from sklearn.decomposition import PCA
from torch_kmeans import KMeans, CosineSimilarity

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap

colors = [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0)]
cmaps = [
    ListedColormap([(1, 0, 0, i / 255) for i in range(255)]),
    ListedColormap([(0, 1, 0, i / 255) for i in range(255)]),
    ListedColormap([(0, 0, 1, i / 255) for i in range(255)]),
    ListedColormap([(1, 1, 0, i / 255) for i in range(255)]),
]

plt.rcParams["savefig.bbox"] = "tight"


def show(imgs):
    if not isinstance(imgs, list):
        imgs = [imgs]
    fix, axs = plt.subplots(ncols=len(imgs), squeeze=False)
    for i, img in enumerate(imgs):
        img = img.detach()
        img = T.functional.to_pil_image(img)
        axs[0, i].imshow(np.asarray(img))
        axs[0, i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

In [ ]:
print(os.getcwd())
os.chdir('/Users/hizlic1/repository-object-centric/ssl_nat_aug/ssl_tests')
# os.chdir("/home/mereur1/projects/ocl/ssl_nat_aug/test_patch_emb")
print(os.getcwd())

In [ ]:
torch.manual_seed(0)
#
ROOT = "/Users/hizlic1/repository-object-centric/ms-coco"
IMAGES_PATH = "/Users/hizlic1/repository-object-centric/ms-coco/val2017"
ANNOTATIONS_PATH = "/Users/hizlic1/repository-object-centric/ms-coco/annotations/instances_val2017.json"

# ROOT = "/home/mereur1/projects/ocl/data/COCO"
# IMAGES_PATH = "/home/mereur1/projects/ocl/data/COCO/val2017"
# ANNOTATIONS_PATH = "/home/mereur1/projects/ocl/data/COCO/annotations/instances_val2017.json"


dataset_untransformed = datasets.CocoDetection(IMAGES_PATH, ANNOTATIONS_PATH)
dataset_untransformed = datasets.wrap_dataset_for_transforms_v2(
    dataset_untransformed,
    target_keys=("boxes", "labels", "masks", "image_id", "segmentation"),
)

In [ ]:
import json

with open(ANNOTATIONS_PATH, "r") as f:
    root = json.load(f)

root.keys()

n_images = len(root["images"])
n_boxes = len(root["annotations"])
n_categories = len(root["categories"])

heights = [x["height"] for x in root["images"]]
widths = [x["width"] for x in root["images"]]

# print('Dataset Name: ',src_desc)
print("Number of images: ", n_images)
print("Number of bounding boxes: ", n_boxes)
print("Number of classes: ", n_categories)
print(
    "Max min avg height: ", max(heights), min(heights), int(sum(heights) / len(heights))
)
print("Max min avg width: ", max(widths), min(widths), int(sum(widths) / len(widths)))

categ_map = {x["id"]: "_".join(x["name"].split()) for x in root["categories"]}
for k in categ_map.keys():
    print(k, "->", categ_map[k], end="\n")

In [ ]:
res = 322
transform = T.Compose(
    [
        T.Resize(res, Image.NEAREST),
        T.CenterCrop(res),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
dataset_val = datasets.CocoDetection(
    root=IMAGES_PATH, annFile=ANNOTATIONS_PATH, transform=transform
)
dataset_val = datasets.wrap_dataset_for_transforms_v2(
    dataset_val, target_keys=["boxes", "labels", "masks", "image_id", "segmentation"]
)
dataloader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=100,
    shuffle=False,
    collate_fn=lambda batch: tuple(zip(*batch)),
)

invTrans = T.Compose(
    [
        T.Normalize(mean=[0.0, 0.0, 0.0], std=[1 / 0.229, 1 / 0.224, 1 / 0.225]),
        T.Normalize(mean=[-0.485, -0.456, -0.406], std=[1.0, 1.0, 1.0]),
    ]
)

In [11]:
for i, batch in enumerate(dataloader_val):
    break

In [ ]:
images, targets = batch
images = torch.stack(images)
print(images.shape)

In [13]:
idx = 1
img, target = dataset_untransformed[idx]
# plt.imshow(img)
# plt.axis('off')
# plt.tight_layout()
# plt.show()

In [ ]:
print([categ_map[l.item()] for l in target["labels"]])
print(type(img), type(target["masks"][0]), target["masks"][0].shape)
# PIL image to torch.tensor
img_tensor = T.ToTensor()(img)
# plt.imshow(img_tensor.permute(1, 2, 0) * target['masks'][0][:, :, None])
# plt.axis('off')
# plt.tight_layout()
# plt.show()

# Intra-Image Patch Similarities

## A Single Object Example

In [ ]:
idx = 1
img, target = dataset_val[idx]
plt.figure(figsize=(10, 5))
plt.subplot(121)
print(img.min(), img.max(), invTrans(img).min(), invTrans(img).max())
plt.imshow(invTrans(img).permute(1, 2, 0))
plt.axis("off")
plt.tight_layout()

stride = 14
h, w = img.shape[1:]
height_int = (h // stride) * stride
width_int = (w // stride) * stride
img_resized = torch.nn.functional.interpolate(
    img.unsqueeze(0), size=(height_int, width_int), mode="bilinear"
)


patches = img_resized.unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
print(patches.shape)
patches = patches.squeeze(0).permute(1, 2, 0, 3, 4)

plt.subplot(122)
plt.imshow(
    make_grid(
        [p for p in invTrans(patches.reshape(-1, 3, 14, 14))],
        nrow=23,
        padding=1,
        pad_value=1.0,
    ).permute(1, 2, 0)
)
plt.axis("off")
plt.show()

#### Pixel similarities

In [16]:
# patch_vector = patches.reshape(23, 23, -1).permute(2, 0, 1).unsqueeze(0)
# print(patch_vector.shape)
# attn_plain = torch.einsum("nchw,ncij->nhwij", F.normalize(patch_vector, dim=1), F.normalize(patch_vector, dim=1))
# attn_plain -= attn_plain.mean([3, 4], keepdims=True)
# attn_plain = attn_plain.clamp(0).squeeze(0)

# plt.figure(figsize=(10,10))
# sns.heatmap(attn_plain[0, 0], annot=True, fmt=".1f", cmap=cmaps[0], linewidths=.5, )
# plt.show()

# plt.figure(figsize=(10,10))
# sns.heatmap(attn_plain[5, 5], annot=True, fmt=".1f", cmap=cmaps[0], linewidths=.5, )
# plt.show()

# plt.figure(figsize=(10,10))
# sns.heatmap(attn_plain[10, 10], annot=True, fmt=".1f", cmap=cmaps[0], linewidths=.5, )
# plt.show()

# plt.figure(figsize=(10,10))
# sns.heatmap(attn_plain[20, 20], annot=True, fmt=".1f", cmap=cmaps[0], linewidths=.5, )
# plt.show()

## Finding 1: DINOv2 does not really compress the image signal

DINOv2 takes the image signal as input $x: H \times W \times C$ and outputs a dense embedding $z: H/14 \times W/14 \times K$.

For example, in the STEGO paper, $x_i \in \mathbb{R}^{322 \times 322 \times 3 \sim 300k}$ and its embedding $z_i \in \mathbb{R}^{23 \times 23 \times 384 \sim 200k}$.

In [ ]:
os.getcwd()

In [ ]:
z = torch.load(
    "/Users/hizlic1/repository-object-centric/ssl_nat_aug/ssl_feats/DINOv2/embeddings_val.pt"
    # "/home/mereur1/projects/ocl/ssl_nat_aug/test_patch_emb/outputs/DINOv2/embeddings_val.pt"
)
z.shape

#### Unnormalized Cosine Similarities

In [20]:
attn_intra = torch.einsum(
    "nchw,ncij->nhwij", F.normalize(z[1:2], dim=1), F.normalize(z[1:2], dim=1)
)
# attn_intra -= attn_intra.mean([3, 4], keepdims=True)
attn_intra = attn_intra.clamp(0).squeeze(0)
# heatmap_intra = F.interpolate(attn_intra, img.shape[1:], mode="bilinear", align_corners=True).squeeze(0).detach().cpu()
# heatmap_intra.shape

# vmax = np.abs(heatmap_intra).max()
# print(vmax)
img_point_h, img_point_w = 20, 20
# plt.figure(figsize=(5,5))
# print(img.min(), img.max(), invTrans(img).min(), invTrans(img).max())
# plt.imshow(invTrans(img).permute(1,2,0))
# plt.scatter(img_point_h*14+7, img_point_w*14+7, c=colors[0], marker="x", s=500, linewidths=5)
# plt.imshow(heatmap_intra[img_point_h, img_point_w], cmap='bwr', alpha=0.5, vmin=-vmax, vmax=vmax)
# plt.axis('off')
# plt.tight_layout()
# plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[0, 0],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[5, 5],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[10, 10],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[20, 20],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()

#### Normalized Cosine Similarities (Mean-subtracted)

In [ ]:
attn_intra = torch.einsum(
    "nchw,ncij->nhwij", F.normalize(z[1:2], dim=1), F.normalize(z[1:2], dim=1)
)
attn_intra -= attn_intra.mean([3, 4], keepdims=True)
attn_intra = attn_intra.clamp(0).squeeze(0)
heatmap_intra = (
    F.interpolate(attn_intra, img.shape[1:], mode="bilinear", align_corners=True)
    .squeeze(0)
    .detach()
    .cpu()
)
heatmap_intra.shape

vmax = np.abs(heatmap_intra).max()
print(vmax)
img_point_h, img_point_w = 20, 20
plt.figure(figsize=(5, 5))
print(img.min(), img.max(), invTrans(img).min(), invTrans(img).max())
plt.imshow(invTrans(img).permute(1, 2, 0))
plt.scatter(
    img_point_h * 14 + 7,
    img_point_w * 14 + 7,
    c=colors[0],
    marker="x",
    s=500,
    linewidths=5,
)
plt.imshow(
    heatmap_intra[img_point_h, img_point_w],
    cmap="bwr",
    alpha=0.5,
    vmin=-vmax,
    vmax=vmax,
)
plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[0, 0],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[5, 5],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[10, 10],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()
plt.figure(figsize=(10, 10))
sns.heatmap(
    attn_intra[20, 20],
    annot=True,
    fmt=".1f",
    cmap=cmaps[0],
    linewidths=0.5,
)
plt.show()

## Multi-Object Example

In [ ]:
img_idx = 10
img, target = dataset_val[img_idx]

# Calculate Intra-image patch embedding similarity
attn_intra = torch.einsum(
    "nchw,ncij->nhwij",
    F.normalize(z[img_idx : img_idx + 1], dim=1),
    F.normalize(z[img_idx : img_idx + 1], dim=1),
)
attn_intra -= attn_intra.mean([3, 4], keepdims=True)
attn_intra = attn_intra.clamp(0).squeeze(0)

print(img.shape)
patches = img.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
patches = patches.squeeze(0).permute(1, 2, 0, 3, 4)
p1, p2, nc, ps, _ = patches.shape

resize_transform = T.Compose([T.Resize(322, Image.NEAREST), T.CenterCrop(322)])
obj_ids, obj_labels, obj_strs = [], [], []
print("All objects before resize", [categ_map[l.item()] for l in target["labels"]])
for i in range(len(target["labels"])):
    obj_mask = target["masks"][i]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float())
    if obj_mask.any():
        obj_ids.append(i)
        obj_labels.append(target["labels"][i].item())
        obj_strs.append(categ_map[target["labels"][i].item()])
print("All objects after resize", obj_strs)
num_obj = len(obj_ids)

ncol = 4
fig = plt.figure(figsize=(5 * ncol, 5 * (num_obj + 1)))
plt.subplot(num_obj + 1, ncol, 1)
plt.imshow(invTrans(img).permute(1, 2, 0))
plt.axis("off")
plt.title("Original Image + Resize + CenterCrop")

plt.subplot(num_obj + 1, ncol, 2)
plt.title("23x23 Patches of size (14x14) (DINOv2)")
plt.axis("off")
plt.imshow(
    make_grid(
        [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
        nrow=p1,
        padding=1,
        pad_value=1.0,
    ).permute(1, 2, 0)
)

for ii, (obj_idx, obj_label, obj_label_str) in enumerate(
    zip(obj_ids, obj_labels, obj_strs)
):
    obj_mask = target["masks"][obj_idx]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float()).repeat(3, 1, 1)
    img_masked = invTrans(img) * obj_mask

    patches_masked = (
        obj_mask.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    )
    patches_masked = patches_masked.squeeze(0).permute(1, 2, 0, 3, 4)
    obj_patch_ids = patches_masked.nonzero()[:, :2].unique(dim=(0))
    idx = torch.randperm(len(obj_patch_ids))[0]
    obj_patch_idx = obj_patch_ids[idx]
    obj_row, obj_col = obj_patch_idx[0].item(), obj_patch_idx[1].item()

    img_masked[obj_mask == 0] = 1.0
    patches_masked = (
        img_masked.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    )
    patches_masked = patches_masked.squeeze(0).permute(1, 2, 0, 3, 4)
    p1, p2, nc, ps, _ = patches_masked.shape

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 1)
    plt.title(f"Segment Object (Id:{obj_label}, Label: {obj_label_str})")
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in patches_masked.reshape(-1, nc, ps, ps)],
            nrow=p1,
            padding=1,
            pad_value=0.0,
        ).permute(1, 2, 0)
    )
    plt.scatter(
        obj_col * (ps + 1) + ps // 2,
        obj_row * (ps + 1) + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.7,
    )
    # plt.title(f'Segment Object (Id:{obj_label}, Label: {obj_label_str})')
    # plt.imshow(img_masked.permute(1,2,0))
    # plt.axis('off')

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 2)
    plt.title(f"Object Patch Sample: {(obj_row, obj_col)}")
    sample_patch = torch.clone(patches_masked)
    for i in range(p1):
        for j in range(p2):
            if i != obj_row or j != obj_col:
                sample_patch[i, j] = 1.0
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in sample_patch.reshape(-1, nc, ps, ps)],
            nrow=p1,
            padding=1,
            pad_value=0.0,
        ).permute(1, 2, 0)
    )

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 3)
    plt.title("All Patches")
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
            nrow=p1,
            padding=1,
            pad_value=1.0,
        ).permute(1, 2, 0)
    )
    plt.scatter(
        obj_col * (ps + 1) + ps // 2,
        obj_row * (ps + 1) + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.7,
    )

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 4)
    plt.title("Intra-Image Similarity Heatmap")
    sns.heatmap(
        attn_intra[obj_row, obj_col],
        annot=True,
        fmt=".1f",
        cmap=cmaps[0],
        linewidths=0.5,
    )

plt.tight_layout()
plt.show()

## Finding 2: Within-image patch similarities catch semantics in smaller objects?
* Depending on the object scale, patches capture object or object part semantics.
* For the bear image, bear object contains ~200 patches. The patches seem to specialize to object parts (ear, face, body).
* For the city image, 1 object contains at most ~15 patches. The patch similarities capture object semantics (boat instances, human instances, bird neck -> other bird parts).

In [ ]:
img_idx = 0
img, target = dataset_val[img_idx]

# Calculate Intra-image patch embedding similarity
attn_intra = torch.einsum(
    "nchw,ncij->nhwij",
    F.normalize(z[img_idx : img_idx + 1], dim=1),
    F.normalize(z[img_idx : img_idx + 1], dim=1),
)
attn_intra -= attn_intra.mean([3, 4], keepdims=True)
attn_intra = attn_intra.clamp(0).squeeze(0)

print(img.shape)
patches = img.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
patches = patches.squeeze(0).permute(1, 2, 0, 3, 4)
p1, p2, nc, ps, _ = patches.shape

resize_transform = T.Compose([T.Resize(322, Image.NEAREST), T.CenterCrop(322)])
obj_ids, obj_labels, obj_strs = [], [], []
print("All objects before resize", [categ_map[l.item()] for l in target["labels"]])
for i in range(len(target["labels"])):
    obj_mask = target["masks"][i]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float())
    if obj_mask.any():
        obj_ids.append(i)
        obj_labels.append(target["labels"][i].item())
        obj_strs.append(categ_map[target["labels"][i].item()])
print("All objects after resize", obj_strs)
num_obj = len(obj_ids)

ncol = 4
fig = plt.figure(figsize=(5 * ncol, 5 * (num_obj + 1)))
plt.subplot(num_obj + 1, ncol, 1)
plt.imshow(invTrans(img).permute(1, 2, 0))
plt.axis("off")
plt.title("Original Image + Resize + CenterCrop")

plt.subplot(num_obj + 1, ncol, 2)
plt.title("23x23 Patches of size (14x14) (DINOv2)")
plt.axis("off")
plt.imshow(
    make_grid(
        [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
        nrow=p1,
        padding=1,
        pad_value=1.0,
    ).permute(1, 2, 0)
)

for ii, (obj_idx, obj_label, obj_label_str) in enumerate(
    zip(obj_ids, obj_labels, obj_strs)
):
    obj_mask = target["masks"][obj_idx]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float()).repeat(3, 1, 1)
    img_masked = invTrans(img) * obj_mask

    patches_masked = (
        obj_mask.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    )
    patches_masked = patches_masked.squeeze(0).permute(1, 2, 0, 3, 4)
    obj_patch_ids = patches_masked.nonzero()[:, :2].unique(dim=(0))
    idx = torch.randperm(len(obj_patch_ids))[0]
    obj_patch_idx = obj_patch_ids[idx]
    obj_row, obj_col = obj_patch_idx[0].item(), obj_patch_idx[1].item()

    img_masked[obj_mask == 0] = 1.0
    patches_masked = (
        img_masked.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    )
    patches_masked = patches_masked.squeeze(0).permute(1, 2, 0, 3, 4)
    p1, p2, nc, ps, _ = patches_masked.shape

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 1)
    plt.title(f"Segment Object (Id:{obj_label}, Label: {obj_label_str})")
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in patches_masked.reshape(-1, nc, ps, ps)],
            nrow=p1,
            padding=1,
            pad_value=0.0,
        ).permute(1, 2, 0)
    )
    plt.scatter(
        obj_col * (ps + 1) + ps // 2,
        obj_row * (ps + 1) + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.7,
    )
    # plt.title(f'Segment Object (Id:{obj_label}, Label: {obj_label_str})')
    # plt.imshow(img_masked.permute(1,2,0))
    # plt.axis('off')

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 2)
    plt.title(f"Object Patch Sample: {(obj_row, obj_col)}")
    sample_patch = torch.clone(patches_masked)
    for i in range(p1):
        for j in range(p2):
            if i != obj_row or j != obj_col:
                sample_patch[i, j] = 1.0
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in sample_patch.reshape(-1, nc, ps, ps)],
            nrow=p1,
            padding=1,
            pad_value=0.0,
        ).permute(1, 2, 0)
    )

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 3)
    plt.title("All Patches")
    plt.axis("off")
    plt.imshow(
        make_grid(
            [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
            nrow=p1,
            padding=1,
            pad_value=1.0,
        ).permute(1, 2, 0)
    )
    plt.scatter(
        obj_col * (ps + 1) + ps // 2,
        obj_row * (ps + 1) + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.7,
    )

    plt.subplot(num_obj + 1, ncol, (ii + 1) * ncol + 4)
    plt.title("Intra-Image Similarity Heatmap")
    sns.heatmap(
        attn_intra[obj_row, obj_col],
        annot=True,
        fmt=".1f",
        cmap=cmaps[0],
        linewidths=0.5,
    )

plt.tight_layout()
plt.show()

## Inter-Image Patch Similarities

In [22]:
num_img = 100
img_grid = [invTrans(dataset_val[i][0]) for i in range(num_img)]
img_grid = make_grid(img_grid, nrow=10)
# plt.figure(figsize=(20, 20))
# plt.imshow(img_grid.permute(1, 2, 0))
# plt.axis('off')

In [ ]:
for seed in range(10):
    torch.manual_seed(seed)

    num_img = 100
    # Sample image
    img_idx = torch.randperm(num_img)[0].item()
    img, target = dataset_val[img_idx]
    print("Image Index:", img_idx)
    # Find objects
    resize_transform = T.Compose([T.Resize(322, Image.NEAREST), T.CenterCrop(322)])
    obj_ids, obj_labels, obj_strs = [], [], []
    print("All objects before resize", [categ_map[l.item()] for l in target["labels"]])
    for i in range(len(target["labels"])):
        obj_mask = target["masks"][i]
        obj_mask = resize_transform(obj_mask.unsqueeze(0).float())
        if obj_mask.any():
            obj_ids.append(i)
            obj_labels.append(target["labels"][i].item())
            obj_strs.append(categ_map[target["labels"][i].item()])
    print("All objects after resize", obj_strs)
    num_obj = len(obj_ids)

    # Sample object
    obj_idx = torch.randperm(num_obj)[0]
    print(
        f"Object Index: {obj_idx}, Label: {obj_labels[obj_idx]}, Name: {obj_strs[obj_idx]}"
    )

    obj_mask = target["masks"][obj_idx]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float()).repeat(3, 1, 1)
    # img_masked = invTrans(img) * obj_mask

    obj_mask_patches = (
        obj_mask.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    )
    obj_mask_patches = obj_mask_patches.squeeze(0).permute(1, 2, 0, 3, 4)
    obj_patch_ids = obj_mask_patches.nonzero()[:, :2].unique(dim=(0))
    idx = torch.randperm(len(obj_patch_ids))[0]
    obj_patch_idx = obj_patch_ids[idx]
    obj_row, obj_col = obj_patch_idx[0].item(), obj_patch_idx[1].item()
    print(f"Object Patch Index: {obj_row, obj_col}")

    # Calculate Inter-Image patch embedding similarity
    z_inter = torch.cat([z[:img_idx], z[img_idx + 1 : num_img]], dim=0)
    attn_inter = torch.einsum(
        "nchw,ncij->nhwij",
        F.normalize(z[img_idx : img_idx + 1].repeat(num_img - 1, 1, 1, 1), dim=1),
        F.normalize(z_inter, dim=1),
    )
    attn_inter -= attn_inter.mean([3, 4], keepdims=True)
    attn_inter = attn_inter.clamp(0).squeeze(0)

    obj_patch_similarities = attn_inter[:, obj_row, obj_col]
    topk_tensor = torch.topk(obj_patch_similarities.flatten(), 10, sorted=True)
    print(topk_tensor.values)
    indices = torch.unravel_index(topk_tensor.indices, obj_patch_similarities.shape)

    topk_images = indices[0].numpy()
    _, idx = np.unique(topk_images, return_index=True)
    topk_images = topk_images[np.sort(idx)]
    topk_images = topk_images.tolist()
    print(topk_images)
    for iii in range(len(topk_images)):
        topk_images[iii] = (
            topk_images[iii] if topk_images[iii] < img_idx else topk_images[iii] + 1
        )
    img_grid = [invTrans(dataset_val[i][0]) for i in topk_images]
    img_W = img_grid[0].shape[2]
    img_grid = make_grid(img_grid, nrow=len(topk_images))

    fig, axs = plt.subplots(
        1,
        2,
        figsize=(5 * (len(topk_images) + 1), 5),
        gridspec_kw={"width_ratios": [1, len(topk_images)], "wspace": 0.1},
    )

    img = dataset_val[img_idx][0]
    patches = img.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
    patches = patches.squeeze(0).permute(1, 2, 0, 3, 4)
    p1, p2, nc, ps, _ = patches.shape

    axs[0].set_title("Source Image and Patch, Label: " + obj_strs[obj_idx])
    axs[0].imshow(
        make_grid(
            [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
            nrow=p1,
            padding=1,
            pad_value=1.0,
        ).permute(1, 2, 0)
    )
    axs[0].scatter(
        obj_col * (ps + 1) + ps // 2,
        obj_row * (ps + 1) + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.7,
    )
    # axs[0].annotate('S', (obj_col*(ps+1)+ps//2, obj_row*(ps+1)+ps//2), color='white', fontsize=12, ha='center', va='center')
    # axs[0].imshow(invTrans(img).permute(1, 2, 0))
    axs[0].axis("off")

    axs[1].set_title("Top-10 Similar Patches")
    axs[1].imshow(img_grid.permute(1, 2, 0))
    for ii, (o, i, j) in enumerate(zip(*indices)):
        o = o.item()
        o = o if o < img_idx else o + 1
        i = i.item()
        j = j.item()
        img_order = topk_images.index(o)
        print(o, i, j, img_order)
        axs[1].scatter(
            (img_order + 1) * 2 + img_order * img_W + j * ps + ps // 2,
            2 + i * ps + ps // 2,
            c=colors[0],
            marker="x",
            s=100,
            linewidths=5,
            alpha=0.4,
        )
        axs[1].annotate(
            f"{ii+1}",
            (
                (img_order + 1) * 2 + img_order * img_W + j * (ps) + ps // 2,
                2 + i * (ps) + ps // 2,
            ),
            color="white",
            fontsize=12,
            ha="center",
            va="center",
            alpha=0.5,
        )

    axs[1].axis("off")

## Finding 3: Patch-level representations are more about object parts, color or textures (such as person hair, pants, elephant boundary) rather than semantics.

## Finding 4: The query patch may include other objects, textures, or background. (skiis and pants and snow)

## Finding 5: Hence, retrieval results depend on the selection of the query patch.

In [24]:
# A = torch.randn(11, 10, 9)
# B = torch.randn(11, 10, 9)

# dist_manual = (A[:, :, None, None, :] - B[None, None, :, :, :]).pow(2).sum(-1).sqrt()


# dist = torch.cdist(A[:, :, None, None, :], B[None, None, :, :, :], p=2)
# dist_broadcast = dist.squeeze(3)
# print(dist_broadcast.shape)

# dists = []
# for i in range(11):
#     col_dists = []
#     for j in range(10):
#         dist = torch.cdist(A[i][j][None, None, :], B, p=2)
#         col_dists.append(dist.squeeze(1))
#     col_dists = torch.stack(col_dists)
#     dists.append(col_dists)

# print(torch.allclose(dist_broadcast, torch.stack(dists), atol=1e-6))
# print(torch.allclose(dist_broadcast, dist_manual, atol=1e-6))

#### Measuring the Similarity by the Euclidean Distance

In [ ]:
torch.manual_seed(4)

num_img = 100
# Sample image
img_idx = torch.randperm(num_img)[0].item()
img, target = dataset_val[img_idx]
print("Image Index:", img_idx)
# Find objects
resize_transform = T.Compose([T.Resize(322, Image.NEAREST), T.CenterCrop(322)])
obj_ids, obj_labels, obj_strs = [], [], []
print("All objects before resize", [categ_map[l.item()] for l in target["labels"]])
for i in range(len(target["labels"])):
    obj_mask = target["masks"][i]
    obj_mask = resize_transform(obj_mask.unsqueeze(0).float())
    if obj_mask.any():
        obj_ids.append(i)
        obj_labels.append(target["labels"][i].item())
        obj_strs.append(categ_map[target["labels"][i].item()])
print("All objects after resize", obj_strs)
num_obj = len(obj_ids)

# Sample object
obj_idx = torch.randperm(num_obj)[0]
print(
    f"Object Index: {obj_idx}, Label: {obj_labels[obj_idx]}, Name: {obj_strs[obj_idx]}"
)

obj_mask = target["masks"][obj_idx]
obj_mask = resize_transform(obj_mask.unsqueeze(0).float()).repeat(3, 1, 1)
# img_masked = invTrans(img) * obj_mask

obj_mask_patches = obj_mask.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
obj_mask_patches = obj_mask_patches.squeeze(0).permute(1, 2, 0, 3, 4)
obj_patch_ids = obj_mask_patches.nonzero()[:, :2].unique(dim=(0))
idx = torch.randperm(len(obj_patch_ids))[0]
obj_patch_idx = obj_patch_ids[idx]
obj_row, obj_col = obj_patch_idx[0].item(), obj_patch_idx[1].item()
print(f"Object Patch Index: {obj_row, obj_col}")

# Calculate Inter-Image patch embedding similarity
z_inter = torch.cat([z[:img_idx], z[img_idx + 1 : num_img]], dim=0).permute(
    0, 2, 3, 1
)  # nhwc
z_repeat = (
    z[img_idx : img_idx + 1]
    .repeat(num_img - 1, 1, 1, 1)
    .permute(0, 2, 3, 1)[:, obj_row, obj_col]
)  # nc
dist = (z_repeat[:, None, None, :] - z_inter[:, :, :, :]).pow(2).sum(-1).sqrt()
print(dist.shape)
# dist -= dist.mean([1, 2], keepdims=True)
# dist = dist.clamp(0).squeeze(0)

obj_patch_similarities = -dist
topk_tensor = torch.topk(obj_patch_similarities.flatten(), 10, sorted=True)
print(topk_tensor.values)
indices = torch.unravel_index(topk_tensor.indices, obj_patch_similarities.shape)

topk_images = indices[0].numpy()
_, idx = np.unique(topk_images, return_index=True)
topk_images = topk_images[np.sort(idx)]
topk_images = topk_images.tolist()
for iii in range(len(topk_images)):
    topk_images[iii] = (
        topk_images[iii] if topk_images[iii] < img_idx else topk_images[iii] + 1
    )

print(topk_images)
img_grid = [invTrans(dataset_val[i][0]) for i in topk_images]
img_W = img_grid[0].shape[2]
img_grid = make_grid(img_grid, nrow=len(topk_images))

fig, axs = plt.subplots(
    1,
    2,
    figsize=(5 * (len(topk_images) + 1), 5),
    gridspec_kw={"width_ratios": [1, len(topk_images)], "wspace": 0.1},
)

img = dataset_val[img_idx][0]
patches = img.unsqueeze(0).unfold(2, 14, 14).unfold(3, 14, 14).squeeze(0)
patches = patches.squeeze(0).permute(1, 2, 0, 3, 4)
p1, p2, nc, ps, _ = patches.shape

axs[0].set_title("Source Image and Patch, Label: " + obj_strs[obj_idx])
axs[0].imshow(
    make_grid(
        [p for p in invTrans(patches.reshape(-1, nc, ps, ps))],
        nrow=p1,
        padding=1,
        pad_value=1.0,
    ).permute(1, 2, 0)
)
axs[0].scatter(
    obj_col * (ps + 1) + ps // 2,
    obj_row * (ps + 1) + ps // 2,
    c=colors[0],
    marker="x",
    s=100,
    linewidths=5,
    alpha=0.7,
)
# axs[0].annotate('S', (obj_col*(ps+1)+ps//2, obj_row*(ps+1)+ps//2), color='white', fontsize=12, ha='center', va='center')
# axs[0].imshow(invTrans(img).permute(1, 2, 0))
axs[0].axis("off")

axs[1].set_title("Top-10 Similar Patches")
axs[1].imshow(img_grid.permute(1, 2, 0))
for ii, (o, i, j) in enumerate(zip(*indices)):
    o = o.item()
    o = o if o < img_idx else o + 1
    i = i.item()
    j = j.item()
    img_order = topk_images.index(o)
    print(o, i, j, img_order)
    axs[1].scatter(
        (img_order + 1) * 2 + img_order * img_W + j * ps + ps // 2,
        2 + i * ps + ps // 2,
        c=colors[0],
        marker="x",
        s=100,
        linewidths=5,
        alpha=0.4,
    )
    axs[1].annotate(
        f"{ii+1}",
        (
            (img_order + 1) * 2 + img_order * img_W + j * (ps) + ps // 2,
            2 + i * (ps) + ps // 2,
        ),
        color="white",
        fontsize=12,
        ha="center",
        va="center",
        alpha=0.5,
    )

axs[1].axis("off")